# Job Ad Cleaning Pipeline

Steps: load raw data -> basic filtering (dropna/duplicates) -> strip HTML -> mask email/phone/URL -> mask organization/person names (NER) -> save final cleaned corpus.

Fixes vs. the earlier version:
- Original `job_id` from the raw data is kept as-is (not overwritten before filtering)
- HTML tags/entities are stripped (was missing before)
- Single consistent dataframe (`df_clean`) used throughout, `listed_time` actually gets dropped


In [24]:
# import packages
import pandas as pd
import numpy as np
import re
import html
import spacy

In [15]:
df_jobpostings = pd.read_parquet("../data/raw/job_data.parquet")

In [16]:
keep_cols = ["job_id", "title", "description", "location", "formatted_experience_level", "listed_time"]
df_clean = df_jobpostings[keep_cols].copy()

# NOTE: we deliberately do NOT overwrite job_id with range(len(df_clean)) here.
# Doing that before filtering makes the ID depend on which rows happen to survive
# dropna/drop_duplicates -- if the cleaning logic ever changes, the same job ad
# could silently end up with a different job_id, breaking any samples/labels
# drawn earlier against an older version of this file.

df_clean = df_clean.dropna(subset=["description"])
df_clean = df_clean.drop_duplicates(subset=["title", "description"])

assert df_clean["job_id"].is_unique, "job_id is not unique after filtering -- check the raw data"

In [17]:
df_clean["listed_date"] = pd.to_datetime(df_clean["listed_time"], unit="ms")
df_clean = df_clean.drop(columns=["listed_time"])

EMAIL detection/phone number/URL


In [18]:
def normalize_and_extract(text):
    if not isinstance(text, str) or not text.strip():
        return text
    text = re.sub(r"\s*[\(\[]\s*at\s*[\)\]]\s*", "@", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*[\(\[]\s*dot\s*[\)\]]\s*", ".", text, flags=re.IGNORECASE)
    EMAIL_PATTERN_LOOSE = r"[a-zA-Z0-9._%+-]+\s*@\s*[a-zA-Z0-9.-]+\s*\.\s*[a-zA-Z]{2,}"
    text = re.sub(EMAIL_PATTERN_LOOSE, "[EMAIL]", text)
    return text

df_clean["text_cleaned"] = df_clean["description"].apply(normalize_and_extract)

In [19]:
US_PHONE_PATTERN = r"(?:\+?1[\s.-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b"

def clean_phone(text):
    if not isinstance(text, str) or not text.strip():
        return text
    return re.sub(US_PHONE_PATTERN, "[PHONE]", text)

df_clean["text_cleaned"] = df_clean["text_cleaned"].apply(clean_phone)

In [20]:
URL_PATTERN = r"https?://\S+|www\.\S+"

def clean_urls(text):
    if not isinstance(text, str) or not text.strip():
        return text
    return re.sub(URL_PATTERN, "[URL]", text)

df_clean["text_cleaned"] = df_clean["text_cleaned"].apply(clean_urls)


NER with Spacy

In [21]:
# en_core_web_trf: transformer-based (RoBERTa) NER model. Same spaCy API as
# en_core_web_lg, so replace_entities_batch below needs no changes. Requires:
#   pip install spacy-transformers
#   python -m spacy download en_core_web_trf
#
# NOTE: disabling tagger/parser/etc. saves little here, since the transformer
# layer itself (not those components) is the actual bottleneck. Expect this to
# run noticeably slower than en_core_web_lg, especially without a GPU.

# spacy.require_gpu()  # uncomment if you have a working CUDA setup -- helps a lot with _trf

nlp_en = spacy.load(
    "en_core_web_trf",
    disable=["tagger", "parser", "attribute_ruler", "lemmatizer"]
)

LABELS_TO_REPLACE = {"ORG": "[ORG]", "PERSON": "[PERSON]"}

def replace_entities_batch(texts, nlp, batch_size=32, n_process=1):
    results = []

    docs = nlp.pipe(texts, batch_size=batch_size, n_process=n_process)

    for text, doc in zip(texts, docs):
        for ent in sorted(doc.ents, key=lambda e: e.start_char, reverse=True):
            if ent.label_ in LABELS_TO_REPLACE:
                placeholder = LABELS_TO_REPLACE[ent.label_]
                text = text[:ent.start_char] + placeholder + text[ent.end_char:]

        cleaned_text = re.sub(r"\s+", " ", text).strip()
        results.append(cleaned_text)

    return results

# n_process is kept at 1: spaCy's multiprocessing (n_process > 1) isn't really
# usable here anyway -- transformer models don't support it well on GPU, and on
# CPU it hits the same Windows "spawn"/Jupyter hangs discussed for en_core_web_lg.
# batch_size=32 is a conservative starting point for a transformer; raise it if
# you have a GPU with enough memory, lower it if you hit out-of-memory errors.
df_clean["text_final"] = replace_entities_batch(
    df_clean["text_cleaned"].fillna("").tolist(),
    nlp_en,
    batch_size=32,
    n_process=1
)


In [ ]:
df_clean["job_id"] = range(len(df_clean))
df_sample = df_clean.sample(n=740, random_state=42).copy()

index_splits = np.array_split(range(len(df_sample)), 4)
batches = [df_sample.iloc[idx].copy() for idx in index_splits]

names = ["Leon", "Philipp", "Leander", "Lars"]
target_cols = ["job_id", "title", "gold_rating", "description", "text_final"]

for i, name in enumerate(names):
    others = [b for j, b in enumerate(batches) if j != i]
    df_person = pd.concat(others, ignore_index=True)
    
    df_person["gold_rating"] = ""
    
    df_person = df_person[target_cols]
    
    df_person.to_excel(f"../data/gold_standard_{name}.xlsx", index=False)

In [23]:
df_clean.head(5)

,job_id,title,description,location,formatted_experience_level,listed_date,text_cleaned,text_final
0,3757940104,Hearing Care Provider,Overview\n\nHearingLife is a national hearing ...,"Little River, SC",Entry level,2023-11-04 09:26:40,Overview\n\nHearingLife is a national hearing ...,[ORG][ORG] is a national hearing care company ...
1,3757940025,Shipping & Receiving Associate 2nd shift (Beav...,Metalcraft of Mayville\nMetalcraft of Mayville...,"Beaver Dam, WI",NaN,2023-11-04 06:40:00,Metalcraft of Mayville\nMetalcraft of Mayville...,[ORG] [ORG] of Mayville is an Equal Opportunit...
2,3757938019,"Manager, Engineering",\nThe TSUBAKI name is synonymous with excellen...,"Bessemer, AL",NaN,2023-11-04 06:40:00,\nThe TSUBAKI name is synonymous with excellen...,The [ORG] name is synonymous with excellence i...
3,3757938018,Cook,descriptionTitle\n\n Looking for a great oppor...,"Aliso Viejo, CA",Entry level,2023-11-04 06:40:00,descriptionTitle\n\n Looking for a great oppor...,descriptionTitle Looking for a great opportuni...
4,3757937095,Principal Cloud Security Architect (Remote),"Job Summary\nAt iHerb, we are on a mission to ...",United States,Mid-Senior level,2023-11-04 09:26:40,"Job Summary\nAt iHerb, we are on a mission to ...","Job Summary At [ORG], we are on a mission to m..."


## Finalize & save

In [ ]:
# Drop intermediate columns, keep the fully cleaned text as 'description'
df_clean_final = df_clean.copy()
df_clean_final = df_clean_final.drop(columns=["description", "text_cleaned"])
df_clean_final = df_clean_final.rename(columns={"text_final": "description"})

df_clean_final.to_parquet("../data/job_ads_cleaned.parquet", index=False, compression="zstd")